# 3 — Multimedia JEPA + Collective Bagging (video, image, file, audio, text)

One JEPA encoder per modality (image patches, video tubes, audio spectrogram, text tokens,
file/byte stream). Each predicts **masked latent targets** from context (I-JEPA style, Assran et al.
2023) instead of reconstructing pixels — cheap, semantic, works on unlabelled BuddyUp media.
A joint predictor fuses context embeddings; the *band protocol from notebook 1* gates when a
modality-tower may join the fused bag (linear-probe accuracy bands + grace period).

Extends `jepa_pose_representation.ipynb` (pose/video lane) to 5 modalities.

In [1]:
import importlib.util, os, pathlib, sys
p = pathlib.Path(os.getcwd()).resolve()
ai = None
while p != p.parent:
    for cand in (p / 'backend' / 'ai_service', p / 'ai_service', p):
        if (cand / 'training').is_dir() and (cand / 'notebooks').is_dir():
            ai = cand; break
    if ai is not None: break
    p = p.parent
if ai is None: raise RuntimeError('ai_service not found')
sys.path.insert(0, str(ai / 'training')); sys.path.insert(0, str(ai))
os.chdir(ai / 'notebooks')
# ── load the repo .env (BUDDY_SCALE, KAGGLE_API_TOKEN, …) BEFORE bootstrap ──
# bootstrap reads BUDDY_SCALE at import time, so this must run first. Uses
# python-dotenv when available, else a tiny built-in parser (Kaggle-safe).
def _find_dotenv(start):
    p = pathlib.Path(start).resolve()
    while p != p.parent:
        f = p / '.env'
        if f.is_file():
            return f
        p = p.parent
    return None

_env_file = _find_dotenv(ai)
try:
    from dotenv import load_dotenv
    load_dotenv(_env_file)
except ImportError:
    if _env_file:
        for _line in _env_file.read_text().splitlines():
            _line = _line.strip()
            if not _line or _line.startswith('#') or '=' not in _line:
                continue
            _k, _, _v = _line.partition('=')
            os.environ.setdefault(_k.strip(), _v.strip().strip('"').strip("'"))
print('[env]', _env_file or 'no .env found', '| BUDDY_SCALE =', os.environ.get('BUDDY_SCALE'),
      '| KAGGLE_API_TOKEN =', 'set' if os.environ.get('KAGGLE_API_TOKEN') else 'missing')
_missing = [m for m in ['torch'] if importlib.util.find_spec(m) is None]
if _missing:
    get_ipython().run_line_magic('pip', 'install -q ' + ' '.join(_missing))
# bootstrap.py reads BUDDY_SCALE at import time; if an earlier run in this
# same kernel cached the module (e.g. before the .env was loaded), drop the
# stale copy so the current environment is honoured.
for _stale in ('training.bootstrap', 'bootstrap'):
    _m = sys.modules.get(_stale)
    if _m is not None and getattr(_m, 'BUDDY_SCALE', None) != os.environ.get('BUDDY_SCALE'):
        sys.modules.pop(_stale, None)
        print(f'[bootstrap] re-importing {_stale} (stale scale cache cleared)')
try:
    from training.bootstrap import *
    CFG = init(scale=os.environ.get('BUDDY_SCALE') or None)
except Exception as e:
    print('[bootstrap] unavailable:', e); CFG = {}
except Exception as e:
    print('[bootstrap] unavailable:', e); CFG = {}
SCALE = CFG.get('scale', os.environ.get('BUDDY_SCALE', 'demo'))
print('scale:', SCALE)

[env] /home/peter/Desktop/Buddy-Up/backend/.env | BUDDY_SCALE = smoke | KAGGLE_API_TOKEN = set


2026-09-16 17:41:07.726960: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-16 17:41:08.749723: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


2026-09-16 17:41:11.882111: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


scale: smoke


In [2]:
import torch, torch.nn as nn, numpy as np
torch.manual_seed(1); np.random.seed(1)
BANDS = [0.37, 0.47, 0.57, 0.67, 0.71, 0.73, 0.77, 0.79, 0.81, 0.85, 0.90]
GRACE = {'smoke': 1, 'demo': 2, 'full': 5}[SCALE]
D = 64  # latent dim

class ModalityJEPA(nn.Module):
    """Context encoder + target (EMA) encoder + predictor. Loss = MSE(predictor(ctx), sg(target))."""
    def __init__(self, in_dim, d=D):
        super().__init__()
        self.ctx = nn.Sequential(nn.Linear(in_dim, 128), nn.LayerNorm(128), nn.GELU(), nn.Linear(128, d))
        self.tgt = nn.Sequential(nn.Linear(in_dim, 128), nn.LayerNorm(128), nn.GELU(), nn.Linear(128, d))
        self.pred = nn.Sequential(nn.Linear(d, 128), nn.GELU(), nn.Linear(128, d))
        self._ema()
    @torch.no_grad()
    def _ema(self, m=0.996):
        for a, b in zip(self.tgt.parameters(), self.ctx.parameters()): a.data.mul_(m).add_(b.data, alpha=1-m)
    def forward(self, x_full, mask_ctx, mask_tgt):
        h = self.ctx(x_full)
        zc = self.pred(h[mask_ctx])
        with torch.no_grad(): zt = self.tgt(x_full[mask_tgt])
        n = min(len(zc), len(zt))
        mse = ((zc[:n] - zt[:n]) ** 2).mean()
        reg = torch.relu(1 - h.std(0).mean())  # anti-collapse: pin unit variance
        return mse + reg
    def embed(self, x, toks=8):  # per-sample pooling over its tokens
        return self.ctx(x).view(-1, toks, D).mean(1)

# 5 towers: image(128-patch feat), video(160-tube), audio(96-mel), text(64-tok), file(64-byte-hist)
towers = {'image': ModalityJEPA(128), 'video': ModalityJEPA(160), 'audio': ModalityJEPA(96),
          'text': ModalityJEPA(64), 'file': ModalityJEPA(64)}
fuse = nn.Linear(5 * D, 10)  # 10-way dummy probe task; replace with BuddyUp taxonomy
print('towers:', list(towers.keys()))
from batch_data import has_batch, batch_meta
_MB = batch_meta() if has_batch() else None
if _MB is not None:  # real modalities declare their own feature dims
    towers = {k: ModalityJEPA(v['feat']) for k, v in _MB['towers'].items()}
    print('REAL towers:', {k: v['feat'] for k, v in _MB['towers'].items()})

towers: ['image', 'video', 'audio', 'text', 'file']
REAL towers: {'image': 3072, 'video': 2304, 'audio': 32000, 'text': 64, 'file': 256}


In [3]:
# Synthetic multimodal batch. Real mapping: image=ViT patches, video=TimeSformer tubes,
# audio=log-mel (AudioSet/VGGSound), text=MiniLM tokens, file=byte histogram of uploads.
_DIRS = {}  # fixed per-modality class directions: each modality carries the label
def _dir(mod, feat, nclass=10):
    if (mod, feat) not in _DIRS:
        _DIRS[(mod, feat)] = torch.randn(nclass, feat, generator=torch.Generator().manual_seed(900 + feat))
    return _DIRS[(mod, feat)]
MODS = [('image', 128, 1.2), ('video', 160, 0.8), ('audio', 96, 0.55), ('text', 64, 0.4), ('file', 64, 0.3)]
def synth_batch(bs=32, nclass=10):
    # Planted label signal, stronger in rich modalities (video/image) and weaker in
    # noisy ones (text/file) — like real data. Towers join different bands and promote.
    y = torch.randint(0, nclass, (bs,))
    return {m: torch.randn(bs*8, f) + _dir(m, f)[y.repeat_interleave(8)] * sg for m, f, sg in MODS}, y
def get_batch(bs=32):
    if _RB is None:
        return synth_batch(bs)
    j = torch.randperm(len(_RB['image_X']))[:bs]  # anchor on image tower
    out = {}
    for k, v in _MB['towers'].items():
        out[k] = _RB[f'{k}_X'][j % len(_RB[f'{k}_X'])].reshape(-1, v['feat'])
    return out, None
from batch_data import load_tensors
_RB = None
if _MB is not None:
    _names = [f'{k}_X' for k in towers] + [f'{k}_y' for k in towers]
    _RB = load_tensors(*_names)
    print('REAL batch:', {k: tuple(_RB[f'{k}_X'].shape) for k in towers})
STEPS = {'smoke': 20, 'demo': 80, 'full': 500}[SCALE]
opts = {k: torch.optim.Adam(v.parameters(), lr=1e-3) for k, v in towers.items()}
opts['fuse'] = torch.optim.Adam(fuse.parameters(), lr=1e-2)  # probe fits fast
probe_acc = {k: 0.0 for k in towers}; joined = {}; ce = nn.CrossEntropyLoss()
heads = {k: nn.Linear(D, 10) for k in towers}  # one honest linear probe per tower
opt_h = {k: torch.optim.Adam(heads[k].parameters(), lr=1e-2) for k in towers}
if _RB is not None:  # real towers have their own class counts
    heads = {k: nn.Linear(D, _MB['towers'][k]['nclass']) for k in towers}
    opt_h = {k: torch.optim.Adam(heads[k].parameters(), lr=1e-2) for k in towers}
for s in range(STEPS):
    X, y = get_batch()
    for k, tw in towers.items():  # self-supervised JEPA step per tower
        tw.train(); opts[k].zero_grad()
        idx = torch.randperm(len(X[k])); m = len(idx) // 2
        loss = tw(X[k][idx], idx[:m], idx[m:]); loss.backward(); opts[k].step(); tw._ema()
    if s % 5 == 0:  # per-tower probes gate band membership; the fused head is the bag
        if _RB is not None:  # REAL: own labels per tower; unaligned batch -> no fused head
            for k in towers:
                yk = _RB[f'{k}_y']; nk = len(yk); v = _MB['towers'][k]
                i1 = torch.randperm(nk)[:256]; i2 = torch.randperm(nk)[:128]
                for _ in range(10):
                    opt_h[k].zero_grad()
                    r1 = _RB[f'{k}_X'][i1].reshape(-1, v['feat']).detach()
                    lh = ce(heads[k](towers[k].embed(r1, v['toks'])), yk[i1])
                    lh.backward(); opt_h[k].step()
                with torch.no_grad():
                    r2 = _RB[f'{k}_X'][i2].reshape(-1, v['feat'])
                    acc_k = (heads[k](towers[k].embed(r2, v['toks'])).argmax(1) == yk[i2]).float().mean().item()
                probe_acc[k] = max(probe_acc[k], acc_k)
                b = max([t for t in BANDS if probe_acc[k] >= t], default=None)
                prev = joined[k][0] if k in joined else None
                if b and (prev is None or b > prev):
                    joined[k] = (b, s)
                    verb = 'PROMOTED to' if prev is not None else 'joins band'
                    print(f'step {s}: tower {k} probe={probe_acc[k]:.3f} {verb} {int(b*100)}%')
            elig = [k for k, (_, s0) in joined.items() if s - s0 >= GRACE]
            print(f'step {s}: real probes (unaligned batch, no fused head) eligible={elig}')
        else:
            Xe, ye = synth_batch()  # fresh batch, so probe acc is an honest signal
            for k in towers:  # fit this tower's probe briefly, then score it
                for _ in range(15):
                    opt_h[k].zero_grad()
                    lh = ce(heads[k](towers[k].embed(X[k]).detach()), y)
                    lh.backward(); opt_h[k].step()
                with torch.no_grad():
                    acc_k = (heads[k](towers[k].embed(Xe[k])).argmax(1) == ye).float().mean().item()
                probe_acc[k] = max(probe_acc[k], acc_k)
                b = max([t for t in BANDS if probe_acc[k] >= t], default=None)
                prev = joined[k][0] if k in joined else None
                if b and (prev is None or b > prev):
                    joined[k] = (b, s)
                    verb = 'PROMOTED to' if prev is not None else 'joins band'
                    print(f'step {s}: tower {k} probe={probe_acc[k]:.3f} {verb} {int(b*100)}%')
            for _ in range(15):  # fused bag head trains on all tower embeddings
                opts['fuse'].zero_grad()
                embs = torch.cat([towers[k].embed(X[k]).detach() for k in towers], -1)
                loss = ce(fuse(embs), y); loss.backward(); opts['fuse'].step()
            with torch.no_grad():
                embs = torch.cat([towers[k].embed(Xe[k]) for k in towers], -1)
                acc = (fuse(embs).argmax(1) == ye).float().mean().item()
            elig = [k for k, (_, s0) in joined.items() if s - s0 >= GRACE]
            print(f'step {s}: fused probe acc={acc:.3f} eligible={elig}')
print('joined:', joined)

REAL batch: {'image': (3254, 3072), 'video': (444, 8, 2304), 'audio': (134, 32000), 'text': (8000, 8, 64), 'file': (20000, 256)}


step 0: tower image probe=0.523 joins band 47%


step 0: tower audio probe=0.414 joins band 37%
step 0: tower text probe=0.867 joins band 85%
step 0: tower file probe=0.414 joins band 37%
step 0: real probes (unaligned batch, no fused head) eligible=[]


step 5: tower image probe=0.570 PROMOTED to 56%


step 5: tower audio probe=0.508 PROMOTED to 47%


step 5: tower text probe=0.945 PROMOTED to 90%
step 5: tower file probe=0.562 PROMOTED to 47%
step 5: real probes (unaligned batch, no fused head) eligible=[]


step 10: tower audio probe=0.602 PROMOTED to 56%
step 10: real probes (unaligned batch, no fused head) eligible=['image', 'text', 'file']


step 15: tower image probe=0.672 PROMOTED to 67%


step 15: real probes (unaligned batch, no fused head) eligible=['audio', 'text', 'file']


joined: {'image': (0.67, 15), 'audio': (0.57, 10), 'text': (0.9, 5), 'file': (0.47, 5)}


In [4]:
# Export fused head + one tower encoder (serving pattern: precompute tower embs, fuse ONNX)
fuse.eval(); dummy = torch.randn(1, 5 * D)
torch.onnx.export(fuse, dummy, '../models/multimodal_fuse.onnx', input_names=['emb'], output_names=['logits'],
    dynamic_axes={'emb': {0: 'batch'}})
print('exported ../models/multimodal_fuse.onnx')

/tmp/ipykernel_26926/3321286368.py:3: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(fuse, dummy, '../models/multimodal_fuse.onnx', input_names=['emb'], output_names=['logits'],


[torch.onnx] Obtain model graph for `Linear(in_features=320, out_features=10, bias=True)` with `torch.export.export(..., strict=False)`...


[torch.onnx] Obtain model graph for `Linear(in_features=320, out_features=10, bias=True)` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
exported ../models/multimodal_fuse.onnx


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


## Data, scrapers vs platforms (notebook 3)

| Modality | Best public data (start here) | BuddyUp first-party |
|---|---|---|
| video | HowTo100M, WebVid-2M, Kinetics-400, Ego4D | LiveKit replays + form videos (needs consent + DVC) |
| image | LAION-400M/5B (filtered), COCO, Food-101 | Post/marketplace uploads |
| audio | AudioSet, VGGSound, LibriSpeech | Live-class audio (coach cues) |
| text | C4 / RedPajama sample, Jigsaw | Posts, comments, meal plans |
| file (bytes) | Malimg/byte-hist tricks, or skip tower until needed | Meal-plan PDFs, receipts |

**Scrapers?** No — for multimodal, raw crawling gives you unlicensed faces/voices/music.
Prefer curated sets above + licensed stock (Pexels/Pixabay APIs) + your own consented uploads.
An internet agent/bot is only justified for *metadata* (e.g. resolving food-product barcodes via
OpenFoodFacts API), never for bulk media harvesting.

| Platform | Offers |
|---|---|
| Meta JEPA refs (facebookresearch/jepa, v-jepa) | Reference code, not a service |
| HuggingFace + `transformers` VideoMAE/AST/CLAP | Tower backbones + dataset hosting |
| TwelveLabs / Clarifai / Replicate | Hosted video+audio tagging if you want to *buy* instead of train |
| W&B / MLflow | Track per-tower probe acc vs band joins |